# 02_Practice: BPE vs WordPiece 실전 분석

이 실습에서는 파이썬의 `transformers` 라이브러리를 사용하여 실제 현업에서 쓰이는 **GPT-2(BPE 기반)**와 **BERT(WordPiece 기반)** 토크나이저를 직접 비교해 봅니다.

In [ ]:
# 필요 라이브러리 설치 (로컬 환경에 없는 경우 주석 해제 후 실행하세요)
# !pip install transformers

In [ ]:
from transformers import GPT2Tokenizer, BertTokenizer

# 1. GPT-2 토크나이저 로드 (BPE 방식)
# GPT-2는 기본적으로 BPE 방식을 사용하며, 공백을 Ġ 기호로 표시합니다.
bpe_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# 2. BERT 토크나이저 로드 (WordPiece 방식)
# BERT는 WordPiece 방식을 사용하며, 서브워드 앞에 ## 기호를 붙입니다.
wp_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

## 비교 1: 신조어/복잡한 영단어 처리

In [ ]:
text_eng = "unbelievably"

print("[원본 단어]:", text_eng)
print("-" * 40)
print("[BPE 토큰 분할]:", bpe_tokenizer.tokenize(text_eng))
print("[WordPiece 토큰 분할]:", wp_tokenizer.tokenize(text_eng))

### 분석 (영단어)
- **BPE**: 빈도 기반이므로 기계적인 바이트 쌍 결합의 흔적이 나타납니다.
- **WordPiece**: 언어학적으로 의미 있는 형태소(예: 접미사 `##ably`) 위주로 쪼개는 성향이 나타나며, 원래 단어를 복원할 때 `##`을 기준으로 합칩니다.

## 비교 2: 한글 처리 (다국어 환경에서의 한계점)

In [ ]:
text_kor = "기계 학습을 좋아합니다."

print("[원본 문장]:", text_kor)
print("-" * 40)
print("[BPE 토큰 분할]:", bpe_tokenizer.tokenize(text_kor))
print("[WordPiece 토큰 분할]:", wp_tokenizer.tokenize(text_kor))

### 분석 (한글)
- **BPE (GPT-2)**: 영문 기반으로 학습되었기 때문에 한글을 만나면 글자를 넘어 자음/모음 유니코드 바이트 수준까지 무참히 쪼개버립니다. `Ġ` 기호는 띄어쓰기를 의미합니다.
- **WordPiece (BERT)**: `bert-base-uncased` 역시 영문 전용이므로 한글 단어를 제대로 처리하지 못하고 쪼개집니다. 만약 다국어 모델(mBERT)이나 KoBERT를 사용했다면 형태소 단위로 깔끔하게 잘리며 `##` 기호가 붙습니다.

## 결론
단순히 '텍스트를 자른다'를 넘어, 알고리즘의 **분할 기준(빈도 vs 정보량)**에 따라 전혀 다른 형태의 덩어리가 생성되는 것을 직접 확인했습니다. 기획자는 프로젝트의 언어 타겟과 모델 특성에 맞춰 적절한 토크나이저를 선택해야 합니다.